# Mixed-protocol performance — ALL sessions

The multi-session companion to `single_session_performance_0.ipynb`. It finds **every session that runs
the same random timeout/banishment protocol** (one punishment icon per board that is randomly a
banishment *or* a timeout — detected from the effect's `options` field), builds the per-collection
dataframe from each `log.json`, and reproduces the single-session plots **pooled across all sessions**.

Log-only, no video. All the per-session logic is reused from `mixed_perf.py` (which mirrors notebook 0),
so a number here means the same thing it does there.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic

sys.path.insert(0, str(Path.cwd()))          # this folder holds mixed_perf.py
import mixed_perf as mp
COLR, ELABEL, TYPES = mp.COLR, mp.ELABEL, mp.TYPES
print('mixed_perf loaded; common ->', mp._COMMON)

try:                       # keep figures in memory so save_report() below can collect them all
    get_ipython().run_line_magic('config', 'InlineBackend.close_figures = False')
except Exception:
    pass

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
# Expected layout on the server:   MAIN_DIR / <mouse> / <session> / log.json
MAIN_DIR = '/path/to/MAIN_DIR'      # <-- root folder that holds the MOUSE folders
VIEW_SCALE = mp.DEFAULT_VIEW_SCALE   # world zoom (not in the log; 0.35 default)
PATTERN = '*/*/log.json'             # <mouse>/<session>/log.json ; falls back to a recursive search

MAIN_DIR = Path(MAIN_DIR)
assert MAIN_DIR.exists(), f'MAIN_DIR does not exist: {MAIN_DIR}'

## 1 — find the mixed-protocol sessions

Every `log.json` under `MAIN_DIR/<mouse>/<session>/` whose board offers the random banish/timeout
punishment. The mouse comes from the log's ID (folder name shown alongside as a cross-check), the
session from its folder. Anything that is a different protocol is skipped.

In [ ]:
found = mp.find_sessions(MAIN_DIR, PATTERN)
SESSIONS = []           # (mouse, session, log, df)
rows = []
for p, log in found:
    mouse, session, mouse_folder = mp.session_label(p, log, main_dir=MAIN_DIR)
    df = mp.build_session_df(log, view_scale=VIEW_SCALE, session=session, mouse=mouse)
    SESSIONS.append((mouse, session, log, df))
    ed = log.get('experiment_data', {})
    rows.append(dict(mouse=mouse, mouse_folder=mouse_folder, session=session,
                     date=str(ed.get('datetime', ''))[:19], n_coll=len(df),
                     reward=int((df.effect == 'single_reward').sum()), banish=int((df.effect == 'banish').sum()),
                     timeout=int((df.effect == 'timeout').sum()), escape=int((df.effect == 'unbanish').sum()),
                     path=str(p.parent)))
inv = pd.DataFrame(rows).sort_values(['mouse', 'date']).reset_index(drop=True)
print(f'{len(SESSIONS)} mixed-protocol session(s) across {inv.mouse.nunique()} mouse/mice')
def _digint(s):
    d = ''.join(ch for ch in str(s) if ch.isdigit()); return int(d) if d else None
if len(SESSIONS) and any(r['mouse_folder'] and _digint(r['mouse']) != _digint(r['mouse_folder']) for r in rows):
    print('  note: log ID and folder name give DIFFERENT animals on some rows -- check mouse vs mouse_folder')
assert len(SESSIONS), 'no mixed-protocol sessions found under MAIN_DIR (check the path / PATTERN / layout)'
inv

## 2 — combined dataframe + per-session summary

`ALL` = every collection of every session in one table (tagged with mouse/session). `SUM` = one row per
session (counts, the three stochastic p-values, win-stay, multiplier-bonus captured, path efficiency).

In [ ]:
ALL = pd.concat([d for _, _, _, d in SESSIONS], ignore_index=True)
SUM = pd.DataFrame([mp.session_summary(log, d) for _, _, log, d in SESSIONS])
print('ALL collections:', ALL.shape, '| sessions:', len(SUM))
OUT = MAIN_DIR / 'mixed_protocol_df'
OUT.mkdir(exist_ok=True)
ALL.to_pickle(OUT / 'all_collections.pkl'); SUM.to_pickle(OUT / 'session_summary.pkl')
print('saved ->', OUT)
SUM.round(3)

## 3 — stochastic p-values across sessions

Left: each session's three comparisons (reward vs all negatives / vs timeout only / vs banishment only)
— a point per session, so you see spread and any learning. Right: the **pooled** result over every
collection of every session (the same bar chart as the single-session notebook's section 11).

In [ ]:
comps = [('p_all', 'vs ALL neg', 0), ('p_timeout', 'vs TIMEOUT', 1), ('p_banish', 'vs BANISH', 2)]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
rng = np.random.default_rng(0)
for key, lbl, xi in comps:
    v = SUM[key].dropna().values
    ax[0].scatter(np.full(len(v), xi) + rng.uniform(-.12, .12, len(v)), v, s=40,
                  color=COLR['single_reward'], edgecolor='k', alpha=.8)
    ax[0].scatter([xi], [np.nanmedian(v)], marker='_', s=900, color='k')
ax[0].axhline(0.05, ls='--', color='r', lw=1, label='p = 0.05')
ax[0].set_xticks([0, 1, 2]); ax[0].set_xticklabels(['reward vs\nALL neg', 'reward vs\nTIMEOUT', 'reward vs\nBANISH'])
ax[0].set_ylabel('stochastic p (per session)'); ax[0].set_title('per session (bar = median)'); ax[0].legend(fontsize=8)
ax[0].set_ylim(-0.02, 1.02)

# pooled over every collection
_ben, _det, chance = mp.pfl.world_opportunity(SESSIONS[0][2])
npos = int((ALL.valence == 'positive').sum())
ntime = int((ALL.effect == 'timeout').sum()); nban = int((ALL.effect == 'banish').sum())
pooled = [('reward vs ALL\n(ban+timeout)', npos, ntime + nban), ('reward vs\nTIMEOUT', npos, ntime),
          ('reward vs\nBANISHMENT', npos, nban)]
for i, (lbl, g_, b_) in enumerate(pooled):
    x = g_ + b_; rate = g_ / x if x else np.nan; p = mp.pfl.prob_at_least(g_, x, chance)
    ax[1].bar(i, rate, color=COLR['single_reward'], width=0.6)
    ax[1].text(i, rate + 0.02, f'{rate:.2f}\np={p:.4f}', ha='center', fontsize=9)
ax[1].axhline(chance, ls='--', color='k', label=f'chance {chance:.2f}')
ax[1].set_xticks(range(3)); ax[1].set_xticklabels([c[0] for c in pooled], fontsize=8)
ax[1].set_ylim(0, 1.08); ax[1].set_ylabel('reward rate (pooled)'); ax[1].set_title(f'pooled over all sessions (n={npos+ntime+nban} choices)')
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4 — collections per effect (pooled)

In [ ]:
order = ['single_reward', 'timeout', 'banish', 'unbanish']
cnt = ALL['effect'].value_counts().reindex(order).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(7, 4))
b = ax.bar(range(len(cnt)), cnt.values, color=[COLR[e] for e in cnt.index])
for bar, v in zip(b, cnt.values): ax.text(bar.get_x()+bar.get_width()/2, v+0.5, str(v), ha='center', fontsize=10)
ax.set_xticks(range(len(cnt))); ax.set_xticklabels([ELABEL[e] for e in cnt.index])
ax.set_ylabel('collections (all sessions)'); ax.set_title(f'collections by effect  (n={len(ALL)}, {len(SESSIONS)} sessions)')
plt.tight_layout(); plt.show()

## 5 — distance & time between collections, by type (pooled box)

In [ ]:
cats = [e for e in ('single_reward', 'banish', 'timeout', 'unbanish') if (ALL[ALL.idx > 0].effect == e).any()]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4)); rng = np.random.default_rng(0)
for a, (col, ylab, sc) in zip(ax, [('dt_prev_ms', 'time since previous (s)', 1e-3),
                                   ('dist_prev', 'distance from previous (wu)', 1.0)]):
    data = [ALL[(ALL.effect == e) & (ALL.idx > 0)][col].dropna().values * sc for e in cats]
    bp = a.boxplot(data, patch_artist=True, showfliers=False, widths=0.6)
    for patch, e in zip(bp['boxes'], cats): patch.set_facecolor(COLR[e]); patch.set_alpha(.75)
    for med in bp['medians']: med.set_color('k')
    for i, v in enumerate(data): a.scatter(np.full(len(v), i+1)+rng.uniform(-.12, .12, len(v)), v, s=10, color='0.2', alpha=.4, zorder=3)
    a.set_xticks(range(1, len(cats)+1)); a.set_xticklabels([ELABEL[e] for e in cats]); a.set_ylabel(ylab)
    a.set_title(ylab.split(' (')[0] + ' by type (pooled)')
plt.tight_layout(); plt.show()

## 6 — path efficiency by type (pooled)

In [ ]:
cats = [e for e in ('single_reward', 'banish', 'timeout', 'unbanish') if (ALL.effect == e).any()]
data = [ALL[ALL.effect == e]['path_efficiency'].dropna().values for e in cats]
fig, ax = plt.subplots(figsize=(8, 4.6)); rng = np.random.default_rng(0)
bp = ax.boxplot(data, patch_artist=True, showfliers=False, widths=0.6)
for patch, e in zip(bp['boxes'], cats): patch.set_facecolor(COLR[e]); patch.set_alpha(.75)
for med in bp['medians']: med.set_color('k')
for i, v in enumerate(data): ax.scatter(np.full(len(v), i+1)+rng.uniform(-.12, .12, len(v)), v, s=12, color='0.2', alpha=.45, zorder=3)
ax.set_xticks(range(1, len(cats)+1)); ax.set_xticklabels([ELABEL[e] for e in cats]); ax.set_ylim(0, 1.02)
ax.set_ylabel('path efficiency (1 = beeline, 0 = wandering)'); ax.set_title('approach path efficiency by collected type (pooled)')
plt.tight_layout(); plt.show()
print(ALL.groupby('effect')['path_efficiency'].agg(['median', 'mean', 'count']).round(3).to_string())

## 7 — occupancy around each collected icon (pooled): samples and milliseconds

Icon-centred occupancy pooled over every collection of that type across all sessions. Top row =
position-sample count; bottom row = total time in ms (each sample weighted by the gap to the next).

In [ ]:
def pooled_offsets(effect, window_s=3.0):
    OX, OY, DT = [], [], []
    for _, _, log, d in SESSIONS:
        ox, oy, dt = mp.collection_offsets(log, d, effect, window_s)
        if ox.size: OX.append(ox); OY.append(oy); DT.append(dt)
    if not OX: return np.array([]), np.array([]), np.array([])
    return np.concatenate(OX), np.concatenate(OY), np.concatenate(DT)

HALF, BINS = 900, 45
fig, axes = plt.subplots(2, 3, figsize=(15, 9.5))
for j, (e, lbl) in enumerate(TYPES):
    ox, oy, dt = pooled_offsets(e); n = int((ALL.effect == e).sum())
    for row, (weights, clabel) in enumerate([(None, 'position samples'), (dt, 'time spent (ms)')]):
        a = axes[row, j]
        if ox.size:
            Hh, xe, ye = np.histogram2d(ox, oy, bins=BINS, range=[[-HALF, HALF], [-HALF, HALF]], weights=weights)
            im = a.imshow(Hh.T, origin='lower', extent=[-HALF, HALF, -HALF, HALF], cmap='magma', aspect='equal')
            fig.colorbar(im, ax=a, fraction=0.046, pad=0.04, label=clabel)
        a.plot(0, 0, marker='*', ms=15, color='cyan', mec='k')
        a.set_title(f'{lbl} — {clabel}  (n={n})'); a.set_xlabel('x - icon (wu)'); a.set_ylabel('y - icon (wu)')
plt.suptitle('pooled occupancy around the collected icon (±3 s, icon at centre)'); plt.tight_layout(); plt.show()

## 8 — heading error toward the collected icon vs time (pooled per type)

In [ ]:
GRID = np.linspace(-8, 0, 80)
def pooled_curves_time(effect):
    C = []
    for _, _, log, d in SESSIONS: C += mp.collected_curves_time(log, d, effect)
    return C
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
for a, (eff, lbl) in zip(axes, TYPES):
    curves = pooled_curves_time(eff); grid_vals = []
    for trel, edeg in curves:
        a.plot(trel, edeg, color=COLR[eff], alpha=0.10, lw=.7)
        grid_vals.append(np.interp(GRID, trel, edeg, left=np.nan, right=np.nan))
    if grid_vals:
        M = np.vstack(grid_vals); ok = np.isfinite(M).sum(0) > 0
        mean = np.full(M.shape[1], np.nan); mean[ok] = np.nanmean(M[:, ok], axis=0)
        a.plot(GRID, mean, color=COLR[eff], lw=3, label='mean')
    a.axhline(90, ls=':', color='0.6'); a.set_ylim(0, 180); a.set_xlabel('time to collection (s)'); a.legend(fontsize=8)
    a.set_title(f'{lbl}  (n={len(curves)} collections)')
axes[0].set_ylabel('heading error (deg)\n0 = facing icon, 180 = away')
plt.suptitle('pooled heading error toward the COLLECTED icon vs time'); plt.tight_layout(); plt.show()

## 9 — heading error vs distance to the collected icon (pooled per type, with hittable cone)

In [ ]:
DMAX = 2000.0
R = float(np.nanmedian([mp.collection_radius(log, d) for _, _, log, d in SESSIONS]))
def pooled_dist(effect):
    DD, EE = [], []
    for _, _, log, d in SESSIONS:
        dd, ee = mp.collected_samples_dist(log, d, effect)
        if dd.size: DD.append(dd); EE.append(ee)
    return (np.concatenate(DD), np.concatenate(EE)) if DD else (np.array([]), np.array([]))
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
for a, (eff, lbl) in zip(axes, TYPES):
    dist, err = pooled_dist(eff)
    if dist.size:
        a.scatter(dist, err, s=3, color=COLR[eff], alpha=0.05)
        mean, edges, _ = binned_statistic(dist, err, statistic='mean', bins=25, range=(0, DMAX))
        a.plot((edges[:-1]+edges[1:])/2, mean, color=COLR[eff], lw=3, label='mean')
    dg = np.linspace(R, DMAX, 200); a.plot(dg, np.degrees(np.arcsin(np.clip(R/dg, 0, 1))), color='k', lw=1.3, alpha=.8, label='hittable cone')
    a.axvline(R, ls='--', color='0.4', lw=1.2); a.axhline(90, ls=':', color='0.6')
    a.set_ylim(0, 180); a.set_xlim(DMAX, 0); a.set_xlabel('distance to collected icon (wu)  [near →]'); a.legend(fontsize=7)
    a.set_title(f'{lbl}  (n={dist.size} samples)')
axes[0].set_ylabel('heading error (deg)\n0 = facing icon, 180 = away')
plt.suptitle(f'pooled heading error vs distance to the COLLECTED icon  (collection radius ~{R:.0f} wu)'); plt.tight_layout(); plt.show()

## 10 — decision-making across sessions (win-stay & multiplier bonus)

Per session: win-stay `P(reward right after a reward)` vs `P(reward after a negative)`, and the % of the
multiplier bonus captured. The pooled win-stay counts transitions **within** each session (never across
the boundary between two sessions).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6)); rng = np.random.default_rng(0)
ws = SUM['win_stay'].values; ab = SUM['p_reward_after_bad'].values
for xi, (v, c) in enumerate([(ws, COLR['single_reward']), (ab, COLR['banish'])]):
    vv = v[np.isfinite(v)]
    ax[0].scatter(np.full(len(vv), xi) + rng.uniform(-.1, .1, len(vv)), vv, s=45, color=c, edgecolor='k', alpha=.8)
    ax[0].scatter([xi], [np.nanmean(vv)], marker='_', s=900, color='k')
# pooled win-stay respecting session boundaries
num = den = nab = dab = 0
for _, _, _, d in SESSIONS:
    seq = d[d.valence.isin(['positive', 'negative'])].valence.eq('positive').values
    if len(seq) > 1:
        cur, nxt = seq[:-1], seq[1:]
        num += nxt[cur].sum(); den += cur.sum(); nab += nxt[~cur].sum(); dab += (~cur).sum()
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(['after a REWARD\n(win-stay)', 'after a NEGATIVE'])
ax[0].set_ylim(0, 1.05); ax[0].set_ylabel('P(next = reward)')
ax[0].set_title(f'win-stay per session (bar = mean)\npooled: {num/den:.2f} vs {nab/dab:.2f}')
bc = SUM['bonus_captured'].dropna().values
ax[1].hist(bc, bins=np.linspace(0, 1, 11), color=COLR['single_reward'], edgecolor='k')
ax[1].axvline(np.nanmean(bc), color='k', ls='--', label=f'mean {np.nanmean(bc):.0%}')
ax[1].set_xlabel('multiplier bonus captured'); ax[1].set_ylabel('# sessions'); ax[1].set_title('multiplier bonus captured per session'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f'pooled win-stay: P(reward|reward)={num/den:.3f} (n={den})  vs  P(reward|negative)={nab/dab:.3f} (n={dab})')

## — export: save the whole report as a PDF or PNGs

Run this **after `Run All`** (a fresh run, so the figures above are not duplicated). `save_report('pdf')`
writes every figure, in order, into one multi-page PDF; `save_report('png')` writes them as individual
PNG files. Pass `path=` to redirect (a `.pdf` file, or a folder for the PNGs).

In [ ]:
_stem = 'mixed_protocol'
def save_report(fmt='pdf', path=None, dpi=130):
    '''Save every figure produced above (in creation order) as ONE PDF, or as individual PNGs.'''
    from matplotlib.backends.backend_pdf import PdfPages
    figs = [plt.figure(n) for n in plt.get_fignums()]
    if not figs:
        print('no open figures -- do "Run All" first, then run this cell'); return
    if fmt == 'pdf':
        p = Path(path) if path else MAIN_DIR / 'mixed_protocol_performance.pdf'
        p.parent.mkdir(parents=True, exist_ok=True)
        with PdfPages(p) as pdf:
            for f in figs: pdf.savefig(f, bbox_inches='tight')
        print(f'saved {len(figs)}-page PDF -> {p}')
    else:
        d = Path(path) if path else MAIN_DIR / 'mixed_protocol_df' / 'report_png'
        d.mkdir(parents=True, exist_ok=True)
        for i, f in enumerate(figs, 1):
            f.savefig(d / f'{_stem}_fig{i:02d}.png', dpi=dpi, bbox_inches='tight')
        print(f'saved {len(figs)} PNGs -> {d}')

# save_report('pdf')      # <- uncomment to write the PDF
# save_report('png')      # <- or the PNGs
print('report export ready: call  save_report("pdf")  or  save_report("png")')